> **Updated November 2026.** Amazon Titan Text, Anthropic Claude 3.x and Stable Diffusion XL reached end of life on Amazon Bedrock (see [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). This notebook now uses [Amazon Nova 2 Lite](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) and [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) / [Claude Sonnet 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html) through cross-Region inference profiles. Claude 4.7+ no longer accepts `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so those parameters were removed from the Claude requests.

## Getting started with the Converse API in Amazon Bedrock

In this notebook, we'll explore the basics of the Converse API in Amazon Bedrock. The Converse or ConverseStream API is a unified structured text API action that allows you simplifying the invocations to Bedrock LLMs, using a universal syntax and message structured prompts for any of the supported model providers.

Let's start by installing or updating boto3. You just need to run this cell the first time.

In [1]:
!pip3 install -qU boto3

In [2]:
import boto3
import sys
print('Running boto3 version:', boto3.__version__)

Running boto3 version: 1.44.3


Let's define the region and models to use. We can also setup our boto3 client.

In [3]:
boto3_session = boto3.session.Session()
region = boto3_session.region_name

print('Using region: ', region)

bedrock = boto3.client(
    service_name = 'bedrock-runtime',
    region_name = region,
    )

MODEL_IDS = [
    "us.amazon.nova-2-lite-v1:0",
    "global.anthropic.claude-haiku-5-5",
    "global.anthropic.claude-sonnet-5-5",
]

Using region:  us-east-1


We're now ready to setup our Converse API action in Bedrock. Note that we use the same syntax for any model, including the messages-formatted prompts, and the inference parameters. Also note that we read the output in the same way independently of the model used.

`inferenceConfig` only carries `maxTokens` here because Claude 4.7+ rejects `temperature` and `topP` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)); Nova accepts them, so add them when you call Nova alone.

Claude 5.5 has [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) on by default: the response can start with a `thinking` block (Invoke) or a `reasoningContent` block (Converse) before the `text` block, so the code below picks the text blocks explicitly instead of reading `content[0]`.

Optionally, we could define additional model specific request fields that are not common across all providers. For more information on this check the Bedrock Converse API documentation.

### Converse for one-shot invocations

In [4]:
def invoke_bedrock_model(client, id, prompt, max_tokens=2000):
    response = ""
    try:
        response = client.converse(
            modelId=id,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            inferenceConfig={
                "maxTokens": max_tokens
            }
            #additionalModelRequestFields={
            #}
        )
    except Exception as e:
        print(e)
        result = "Model invocation error"
    try:
        result = "".join(block["text"] for block in response['output']['message']['content'] if "text" in block) \
        + '\n--- Latency: ' + str(response['metrics']['latencyMs']) \
        + 'ms - Input tokens:' + str(response['usage']['inputTokens']) \
        + ' - Output tokens:' + str(response['usage']['outputTokens']) + ' ---\n'
        return result
    except Exception as e:
        print(e)
        result = "Output parsing error"
    return result

Finally, we can test our invocation.

In this example, we run the same prompt across all the text models supported in Bedrock by the time of writing this example.

In [5]:
prompt = ("What is the capital of Italy?")
print(f'Prompt: {prompt}\n')

for i in MODEL_IDS:
    response = invoke_bedrock_model(bedrock, i, prompt)
    print(f'Model: {i}\n{response}')

Prompt: What is the capital of Italy?



Model: us.amazon.nova-2-lite-v1:0
The capital of Italy is **Rome**. 

Rome is not only the capital but also one of the most historically significant cities in the world. It is known as the "Eternal City" and the "City of Seven Hills." It serves as the political, economic, and cultural hub of Italy and is home to numerous landmarks, including the Vatican City (the spiritual and administrative center of the Roman Catholic Church), the Colosseum, the Roman Forum, and many museums and historic sites.
--- Latency: 902ms - Input tokens:53 - Output tokens:99 ---



Model: global.anthropic.claude-haiku-5-5
The capital of Italy is **Rome**.
--- Latency: 966ms - Input tokens:18 - Output tokens:16 ---



Model: global.anthropic.claude-sonnet-5-5
The capital of Italy is **Rome** (Roma in Italian). It is also the country's largest city and has been a major political and cultural center for over 2,500 years.
--- Latency: 1867ms - Input tokens:18 - Output tokens:53 ---



### ConverseStream for streaming invocations

We can also use the Converse API for streaming invocations. In this case we rely on the ConverseStream action.

In [6]:
MODEL_IDS = [
    "us.amazon.nova-2-lite-v1:0",
    "global.anthropic.claude-haiku-5-5",
    "global.anthropic.claude-sonnet-5-5",
]

In [7]:
def invoke_bedrock_model_stream(client, id, prompt, max_tokens=2000):
    response = ""
    response = client.converse_stream(
        modelId=id,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "text": prompt
                    }
                ]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens
        }
    )
    # Extract and print the response text in real-time.
    for event in response['stream']:
        if 'contentBlockDelta' in event:
            chunk = event['contentBlockDelta']
            if 'text' not in chunk['delta']:  # reasoningContent deltas carry no text
                continue
            sys.stdout.write(chunk['delta']['text'])
            sys.stdout.flush()
    return

In [8]:
prompt = ("What is the capital of Italy?")
print(f'Prompt: {prompt}\n')

for i in MODEL_IDS:
    print(f'\n\nModel: {i}')
    invoke_bedrock_model_stream(bedrock, i, prompt)

Prompt: What is the capital of Italy?



Model: us.amazon.nova-2-lite-v1:0


The capital of Italy is **

Rome**.

 

Rome is

 not

 only the capital

... (42 outputs left out)

 major political and cultural center for

 over

 two

 thousand years.

As you can see, the Converse API allow us to easily run the invocations with the same syntax across all the models.